# 🚀 Stage 2: Tabular Classifier Training (XGBoost + Walk-Forward Split + SHAP + ONNX)
**Project**: AI Meme Coin Prediction System (Solana / pump.fun)
**Goal**: Train an ultra-fast XGBoost tabular model on 23 structured features to identify meme coins likely to reach 10x ($5M+ market cap) within 60 minutes of launch.

---
### Architectural & Stress-Test Directives:
1. **Walk-Forward Split (Pass 1 Risk #17)**: Split strictly by time (`launched_at`). Never use random k-fold across market regimes.
   - **Train**: First 70% of chronological window (e.g. Days 0–63)
   - **Validation**: Next 15% (Days 64–76) — hyperparameter tuning, `scale_pos_weight` selection & Platt calibration
   - **Held-Out Test**: Final 15% (Days 77–90) — uncorrupted out-of-time evaluation
2. **Class Imbalance & Base Rate (Pass 3 Risk #38)**: Positive rate is ~0.8% (`scale_pos_weight = 99`, tuned over `[50, 120]`). Evaluation prioritizes **PR-AUC** and **Precision@Top-5** over ROC-AUC.
3. **Zero-Latency Cluster Features (Pass 3 Risk #40)**: `cluster_sniper_count` and `cluster_sniper_supply_pct` are evaluated with SHAP to confirm early bundler penalty.
4. **Production In-Process ONNX Export**: Model is exported to `ml/models/xgb_model.onnx` for sub-millisecond execution in Node.js.

In [ ]:
# Step 1: Install Required Libraries
!pip install -q xgboost scikit-learn shap onnxmltools onnx onnxruntime matplotlib seaborn

In [ ]:
# Step 2: Load Historical Labeled Dataset
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV
try:
    from sklearn.frozen import FrozenEstimator
except ImportError:
    FrozenEstimator = None
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as ort

data_path = '../data/labeled_tokens_historical.jsonl'
assert os.path.exists(data_path), f'Missing dataset: {data_path}'

df = pd.read_json(data_path, lines=True)
df = df.sort_values('launched_at').reset_index(drop=True)
print(f'Loaded {len(df)} historical tokens. Date range: {df["launched_at"].min()} to {df["launched_at"].max()}')

## Step 3: Feature Matrix & Walk-Forward Time Split
Strict time-based split: Train (70%), Validation (15%), Test (15%).

In [ ]:
feature_cols = [
    'mktCapK', 'liquidityK', 'volumeK', 'netBuyK', 'buySellRatio',
    'bCurvePercent', 'bCurveVelocity', 'ageMinutes', 'devRugPercent',
    'devTotalLaunches', 'holdersCount', 'watchersCount', 'watchersDelta',
    'hasSocialLinks', 'hasWebsite', 'isCTO', 'isGraduated', 'txCount',
    'uniqueBuyerRatio', 'cluster_sniper_count', 'cluster_sniper_supply_pct',
    'fee_regime_0', 'fee_regime_1', 'fee_regime_2'
]

X = df[feature_cols].values.astype(np.float32)
y = df['label'].values.astype(int)

n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.85)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_val], y[n_train:n_val]
X_test, y_test = X[n_val:], y[n_val:]

print(f'Train set: {len(X_train)} samples ({y_train.sum()} positives, {y_train.mean()*100:.2f}%)')
print(f'Val set:   {len(X_val)} samples ({y_val.sum()} positives, {y_val.mean()*100:.2f}%)')
print(f'Test set:  {len(X_test)} samples ({y_test.sum()} positives, {y_test.mean()*100:.2f}%)')

## Step 4: Hyperparameter Tuning & `scale_pos_weight` Selection
Given empirical positive rate of ~0.8%, we evaluate `scale_pos_weight` across `[50, 75, 99, 120]` on the validation set.

In [ ]:
candidates = [50, 75, 99, 120]
best_spw = 99
best_val_prauc = -1.0

for spw in candidates:
    clf_test = xgb.XGBClassifier(
        n_estimators=120,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=42,
        eval_metric='logloss'
    )
    clf_test.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_probs = clf_test.predict_proba(X_val)[:, 1]
    p, r, _ = precision_recall_curve(y_val, val_probs)
    pr_auc_val = auc(r, p)
    print(f'scale_pos_weight={spw:3d} -> Validation PR-AUC: {pr_auc_val:.4f}')
    if pr_auc_val > best_val_prauc:
        best_val_prauc = pr_auc_val
        best_spw = spw

print(f'Optimal scale_pos_weight selected: {best_spw}')

# Final Model Training
clf = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=best_spw,
    random_state=42,
    eval_metric='logloss'
)
clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

## Step 5: Platt Calibration on Validation Fold
Per system architecture (Pass 1 risk #19), Platt scaling (`sigmoid`) is mandated until 1,000+ positive samples are accumulated to avoid isotonic overfitting.

In [ ]:
if FrozenEstimator is not None:
    calibrator = CalibratedClassifierCV(estimator=FrozenEstimator(clf), method='sigmoid')
else:
    calibrator = CalibratedClassifierCV(estimator=clf, method='sigmoid', cv='prefit')

calibrator.fit(X_val, y_val)
print('Platt calibration successfully fit on validation fold.')

## Step 6: Out-of-Time Held-Out Test Evaluation
We evaluate performance on the future 15% test set across PR-AUC, ROC-AUC, Brier score, and Precision@Top-5.

In [ ]:
test_probs = calibrator.predict_proba(X_test)[:, 1]

prec, rec, _ = precision_recall_curve(y_test, test_probs)
test_pr_auc = auc(rec, prec)
test_roc_auc = roc_auc_score(y_test, test_probs)
test_brier = brier_score_loss(y_test, test_probs)

top5_idx = np.argsort(test_probs)[-5:]
prec_top5 = np.mean(y_test[top5_idx])

top10_idx = np.argsort(test_probs)[-10:]
prec_top10 = np.mean(y_test[top10_idx])

print('=== Out-of-Time Test Evaluation ===')
print(f'PR-AUC:           {test_pr_auc:.4f}')
print(f'ROC-AUC:          {test_roc_auc:.4f}')
print(f'Brier Score:      {test_brier:.4f}')
print(f'Precision@Top-5:  {prec_top5:.2f}')
print(f'Precision@Top-10: {prec_top10:.2f}')

plt.figure(figsize=(8, 5))
plt.plot(rec, prec, label=f'Platt-Calibrated XGB (PR-AUC={test_pr_auc:.3f})', color='purple', lw=2)
plt.axhline(y_test.mean(), color='grey', linestyle='--', label=f'Baseline Base Rate ({y_test.mean()*100:.2f}%)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Out-of-Time Precision-Recall Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 7: Feature Attribution & SHAP Explainability
Verifies whether zero-latency cluster features (`cluster_sniper_supply_pct`, `cluster_sniper_count`) and developer rug history (`devRugPercent`) properly drive predictions.

In [ ]:
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test[:500])

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test[:500], feature_names=feature_cols, show=False)
plt.title('SHAP Feature Importance (Out-of-Time Test Sample)')
plt.tight_layout()
plt.show()

## Step 8: Production In-Process ONNX Export & Latency Benchmarking
Exports model to `ml/models/xgb_model.onnx` and measures in-process execution speed.

In [ ]:
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)
onnx_path = os.path.join(models_dir, 'xgb_model.onnx')

initial_type = [('float_input', FloatTensorType([None, len(feature_cols)]))]
onnx_model = onnxmltools.convert_xgboost(clf, initial_types=initial_type, target_opset=12)

with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

print(f'Exported XGBoost ONNX model to {onnx_path} ({os.path.getsize(onnx_path)} bytes)')

# In-process ONNX Runtime verification & latency test
import time
session = ort.InferenceSession(onnx_path)
input_name = session.get_inputs()[0].name

# Benchmark 100 single-token scoring calls (simulating live Gate 1)
single_sample = X_test[0:1]
times = []
for _ in range(100):
    t0 = time.perf_counter()
    session.run(None, {input_name: single_sample})
    times.append((time.perf_counter() - t0) * 1000.0)

median_ms = np.median(times)
p99_ms = np.percentile(times, 99)
print(f'ONNX Latency Benchmark (100 single-token runs):')
print(f'Median: {median_ms:.2f}ms | p99: {p99_ms:.2f}ms (Target: < 1.0ms)')